# Haptic Theremin

Y axis = pitch (frets snap to scale notes via spring)
X axis = volume (spring pushes back to ~20%; you push right to get louder)

Run cells top to bottom. Click **▶ Start** then **🔊 Enable Audio**.

In [1]:
import io, threading, time, math
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as w
from IPython.display import display
from sidewinder.ff2 import FF2Device, ET_SPRING, ET_DAMPER

ff = FF2Device()
print('Connected:', ff)

Connected: <FF2Device VID=0x045e PID=0x001b>


In [2]:
# ── Stop any previous session cleanly ────────────────────────────────────────
if '_th_stop' in vars() and _th_stop:
    _th_stop.set(); time.sleep(0.08)
if '_th_fx' in vars():
    for e in _th_fx.values():
        if e:
            try: e.free()
            except: pass

# ── Scales ────────────────────────────────────────────────────────────────────
SCALES = {
    'Major':          [0, 2, 4, 5, 7, 9, 11],
    'Natural Minor':  [0, 2, 3, 5, 7, 8, 10],
    'Pentatonic Maj': [0, 2, 4, 7, 9],
    'Pentatonic Min': [0, 3, 5, 7, 10],
    'Blues':          [0, 3, 5, 6, 7, 10],
    'Dorian':         [0, 2, 3, 5, 7, 9, 10],
    'Chromatic':      list(range(12)),
}
NOTES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

def midi_freq(m): return 440. * 2**((m - 69) / 12)

def build_frets(root, intervals, base_oct, n_oct):
    raw = []
    for o in range(base_oct, base_oct + n_oct + 1):
        for i in intervals:
            m = 12*(o+1) + root + i
            raw.append((midi_freq(m), NOTES[(root+i)%12] + str(o)))
    seen, out = set(), []
    for f, n in sorted(raw, key=lambda x: x[0]):
        k = round(f)
        if k not in seen: seen.add(k); out.append((f, n))
    return out

# ── Controls ──────────────────────────────────────────────────────────────────
S = dict(style={'description_width': '82px'}, layout=w.Layout(width='270px'))
scale_dd  = w.Dropdown(options=list(SCALES), value='Major',    description='Scale:', **S)
root_dd   = w.Dropdown(options=[(n,i) for i,n in enumerate(NOTES)], value=0,
                        description='Root:', style={'description_width':'82px'},
                        layout=w.Layout(width='210px'))
oct_sl    = w.IntSlider(value=3, min=1, max=5,   description='Base oct:', **S)
range_sl  = w.IntSlider(value=2, min=1, max=3,   description='Octaves:',  **S)
spring_sl = w.IntSlider(value=80, min=0, max=127, description='Spring Y:', **S)
damp_sl   = w.IntSlider(value=90, min=0, max=127, description='Damper:',   **S)
xspr_sl   = w.IntSlider(value=80, min=0, max=127, description='Spring X:', **S)
play_btn  = w.ToggleButton(value=False, description='▶ Start',
                            button_style='success', layout=w.Layout(width='100px'))

# State bridge: Python updates data-attrs; JS watches via MutationObserver
state_w = w.HTML(value='<div id="th-state" data-freq="440.0" data-vol="0.0"></div>')

audio_html = w.HTML(value="""
<div style="margin:4px 0 2px">
  <button id="th-btn"
    style="padding:5px 14px;background:#1a7744;color:#fff;border:none;
           border-radius:4px;cursor:pointer;font-size:12px"
    onclick="thereminStartAudio()">
    &#x1F50A; Enable Audio
  </button>
  <span id="th-st" style="margin-left:9px;color:#556;font-size:11px">click first</span>
</div>
<script>(function() {
  var ctx, g, o1, o2, o3;
  function sf(f) {
    if (!ctx || !isFinite(f) || f <= 0) return;
    var t = ctx.currentTime;
    o1.frequency.setTargetAtTime(f,   t, .015);
    o2.frequency.setTargetAtTime(f*2, t, .015);
    o3.frequency.setTargetAtTime(f*3, t, .015);
  }
  function mko(amp) {
    var o = ctx.createOscillator(), gg = ctx.createGain();
    gg.gain.value = amp; o.type = 'sine'; o.connect(gg); gg.connect(g); o.start();
    return o;
  }
  function readState() {
    var el = document.getElementById('th-state');
    if (!el || !ctx) return;
    var freq = parseFloat(el.dataset.freq);
    var vol  = parseFloat(el.dataset.vol);
    if (isFinite(freq) && freq > 0) sf(freq);
    if (isFinite(vol)) g.gain.setTargetAtTime(vol * 0.55, ctx.currentTime, .06);
  }
  function attachObserver() {
    var el = document.getElementById('th-state');
    if (!el) { setTimeout(attachObserver, 300); return; }
    new MutationObserver(readState).observe(el.parentElement, {childList: true, subtree: true});
  }
  window.thereminStartAudio = function() {
    if (ctx) return;
    ctx = new (window.AudioContext || window.webkitAudioContext)();
    g = ctx.createGain(); g.gain.value = 0; g.connect(ctx.destination);
    o1 = mko(.60); o2 = mko(.22); o3 = mko(.10);
    readState();
    attachObserver();
    document.getElementById('th-btn').textContent = '♪ Audio On';
    document.getElementById('th-btn').style.background = '#115577';
    document.getElementById('th-st').textContent = 'running';
  };
  attachObserver();
})();</script>""")

# ── Figure ────────────────────────────────────────────────────────────────────
BG = '#070710'
fig, (ax_v, ax_t) = plt.subplots(1, 2, figsize=(4.5, 5.2), dpi=100,
                                  gridspec_kw={'width_ratios': [1, 5]})
fig.patch.set_facecolor(BG)
for ax in (ax_v, ax_t): ax.set_facecolor(BG)

# Vol strip
ax_v.set_xlim(0,1); ax_v.set_ylim(0,1); ax_v.axis('off')
ax_v.text(0.5, 1.025, 'VOL', color='#334455', fontsize=8, ha='center',
           transform=ax_v.transAxes)
ax_v.add_patch(patches.Rectangle((.25,0), .5, 1, color='#0d1220', zorder=1))
_vbar = patches.Rectangle((.25,0), .5, .20, facecolor='#22bb44', zorder=2)
ax_v.add_patch(_vbar)
ax_v.plot([0.15, 0.85], [0.20, 0.20], color='#334455', lw=1.2, ls='--', zorder=3)
_vpct = ax_v.text(0.5, -0.03, '20%', color='#334455', fontsize=7.5, ha='center')

# Theremin strip
ax_t.set_xlim(-0.5, 0.5); ax_t.set_ylim(-1.06, 1.06); ax_t.axis('off')
ax_t.set_title('Haptic Theremin', color='#7788bb', fontsize=11, pad=5)

_flines, _flabels = [], []
_fret_hi, = ax_t.plot([-0.44,0.44],[0,0], color='#ffcc33', lw=2.2, alpha=0, zorder=5)
_dot,     = ax_t.plot([0],[0], 'o', color='#ff5533', ms=14,
                        mec='#ffaa77', mew=1.8, zorder=10)
_glow,    = ax_t.plot([0],[0], 'o', color='#ff2200', ms=34, alpha=.12, zorder=9)
_nlbl      = ax_t.text(0.46, 0, '', color='#ffee77', fontsize=14,
                        ha='right', va='center', fontweight='bold', zorder=11)
_stxt      = fig.text(0.5, 0.002, '', color='#4455aa', fontsize=8.5, ha='center')

fig.tight_layout(pad=0.4)
_img_w = w.Image(format='png')
_lbl_w = w.Label('stopped')

def _render():
    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor=BG)
    return buf.getvalue()

# Shared state (written by main thread, read by game loop)
_G = dict(frets=[], fy=np.array([0.]), f_min=130., log_span=1.,
          stop=threading.Event())
_th_stop = _G['stop']
_th_fx   = dict(spring=None, damper=None)

def _rebuild():
    for l in _flines:  l.remove()
    for l in _flabels: l.remove()
    _flines.clear(); _flabels.clear()

    raw = build_frets(root_dd.value, SCALES[scale_dd.value],
                      oct_sl.value, range_sl.value)
    f_lo, f_hi = raw[0][0], raw[-1][0]
    ls = math.log(f_hi / f_lo) if f_hi > f_lo else 1.

    fy_list, fret_list = [], []
    for freq, name in raw:
        t = math.log(freq / f_lo) / ls
        y = 2*t - 1           # y=-1 → lowest (bottom), y=+1 → highest (top)
        fy_list.append(y)
        fret_list.append((y, freq, name))
        ln = ax_t.plot([-0.44, 0.44], [y, y], color='#1e2d44', lw=0.9,
                        alpha=0.80, zorder=3)[0]
        lb = ax_t.text(0.47, y, name, color='#445566', fontsize=7.5,
                        ha='right', va='center', zorder=4)
        _flines.append(ln); _flabels.append(lb)

    _G.update(frets=fret_list, fy=np.array(fy_list), f_min=f_lo, log_span=ls)
    _img_w.value = _render()

_rebuild()

# ── Force feedback effects ────────────────────────────────────────────────────
def _start_fx():
    _stop_fx()
    sp = ff.alloc(ET_SPRING)
    sp.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=255)
    sp.set_condition(axis=1, cp_offset=0,
                     pos_coeff=spring_sl.value, neg_coeff=spring_sl.value,
                     pos_sat=200, neg_sat=200, dead_band=0)
    sp.set_condition(axis=0, cp_offset=-76,
                     pos_coeff=xspr_sl.value, neg_coeff=0,
                     pos_sat=200, neg_sat=0, dead_band=0)
    sp.play(); _th_fx['spring'] = sp

    dm = ff.alloc(ET_DAMPER)
    dm.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=255)
    for ax in (0, 1):
        dm.set_condition(axis=ax, pos_coeff=damp_sl.value, neg_coeff=damp_sl.value,
                         pos_sat=255, neg_sat=255, dead_band=0)
    dm.play(); _th_fx['damper'] = dm

def _stop_fx():
    for k, e in list(_th_fx.items()):
        if e:
            try: e.free()
            except: pass
        _th_fx[k] = None

# ── Game loop ─────────────────────────────────────────────────────────────────
def _loop():
    pos = (0., 0.)
    frame = 0
    prev_fi = -1

    while not _G['stop'].is_set():
        try:
            r = ff.read_position()
            if r is not None: pos = r
            x, y = pos

            vol  = max(0., min(1., (x + 1) / 2))
            t    = (1 - y) / 2
            freq = _G['f_min'] * math.exp(t * _G['log_span'])

            fy = _G['fy']; frets = _G['frets']
            if len(fy) and len(frets):
                # frets in display space (y=+1=top=high), joystick y is hardware-inverted
                fi = int(np.argmin(np.abs(fy + y)))
                fi = min(fi, len(frets) - 1)
                fret_y, _, fret_name = frets[fi]
                if fi != prev_fi:
                    sp = _th_fx['spring']
                    if sp:
                        cp = int(np.clip(-fret_y * 127, -127, 127))  # back to joystick space
                        sp.set_condition(axis=1, cp_offset=cp,
                                         pos_coeff=spring_sl.value,
                                         neg_coeff=spring_sl.value,
                                         pos_sat=200, neg_sat=200, dead_band=0)
                    prev_fi = fi
            else:
                fret_y = 0.; fret_name = '—'

            state_w.value = f'<div id="th-state" data-freq="{freq:.2f}" data-vol="{vol:.4f}"></div>'

            frame += 1
            if frame % 3 == 0:
                _dot.set_data([0], [-y])    # negate: joystick up (-y) → dot at top
                _glow.set_data([0], [-y])
                _fret_hi.set_ydata([fret_y, fret_y])
                _fret_hi.set_alpha(0.55)
                _nlbl.set_text(fret_name)
                _nlbl.set_position((0.46, fret_y))
                c = '#22bb44' if vol < .5 else ('#ccaa22' if vol < .8 else '#cc3322')
                _vbar.set_height(vol); _vbar.set_facecolor(c)
                _vpct.set_text(f'{int(vol*100)}%')
                _stxt.set_text(
                    f'{fret_name}  {freq:.1f} Hz   '
                    f'vol {int(vol*100)}%   x={x:+.2f}  y={y:+.2f}')
                _img_w.value = _render()
                _lbl_w.value = f'{fret_name}  {freq:.1f} Hz  {int(vol*100)}%'

        except Exception as ex:
            _lbl_w.value = f'err: {ex}'; break
        time.sleep(0.02)

def _on_play(chg):
    if chg['new']:
        _G['stop'].clear()
        _rebuild()
        _start_fx()
        threading.Thread(target=_loop, daemon=True, name='theremin').start()
        play_btn.description = '⏹ Stop'; play_btn.button_style = 'danger'
    else:
        _G['stop'].set(); time.sleep(0.06); _stop_fx()
        state_w.value = '<div id="th-state" data-freq="440.0" data-vol="0.0"></div>'
        play_btn.description = '▶ Start'; play_btn.button_style = 'success'
        _lbl_w.value = 'stopped'

for ctrl in (scale_dd, root_dd, oct_sl, range_sl):
    ctrl.observe(lambda _: _rebuild(), names='value')

play_btn.observe(_on_play, names='value')

display(w.VBox([
    w.HBox([scale_dd, root_dd]),
    w.HBox([oct_sl, range_sl]),
    w.HBox([spring_sl, damp_sl, xspr_sl]),
    w.HBox([play_btn, audio_html, state_w]),
    _img_w, _lbl_w,
]))

In [3]:
# Run to clean up when done
_G['stop'].set()
_stop_fx()
ff.close()
print('closed')

NameError: name 'ff' is not defined